# 10 - Model Comparison
Side-by-side comparison of logistic regression, random forest, and XGBoost on Brier score, calibration quality, and per-season accuracy.

In [ ]:
import sys
from pathlib import Path

project_root = str(Path.cwd().parent) if Path.cwd().name == 'notebooks' else str(Path.cwd())
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.models.evaluate import (
    compute_brier_scores,
    compute_naive_baseline_brier,
    get_calibration_data,
    plot_calibration_curves,
    plot_brier_by_season,
)

In [ ]:
RESULTS_PATH = Path(project_root) / 'data' / 'results' / 'backtest_results.parquet'
assert RESULTS_PATH.exists(), f"Results not found at {RESULTS_PATH}. Run notebook 09 first."
results_df = pd.read_parquet(RESULTS_PATH)
print(f"Loaded {len(results_df)} predictions")

## Aggregate Brier Scores
Lower Brier score = better calibrated probability predictions. Perfect prediction = 0.0, coin flip = 0.25.

In [ ]:
brier_df = compute_brier_scores(results_df)
agg = brier_df[brier_df['fold_test_year'] == 'aggregate'].sort_values('brier_score')
baseline = compute_naive_baseline_brier(results_df)
baseline_row = pd.DataFrame([baseline])
agg_display = pd.concat(
    [agg[['model_name', 'feature_set', 'brier_score', 'n_games']],
     baseline_row[['model_name', 'feature_set', 'brier_score', 'n_games']]],
    ignore_index=True,
).sort_values('brier_score')
print("Aggregate Brier Scores (lower is better; naive baseline = always predict home win rate ~53.5%):")
print(agg_display.to_string(index=False))

## Per-Season Brier Scores

In [ ]:
per_season = brier_df[brier_df['fold_test_year'] != 'aggregate'].copy()
per_season['fold_test_year'] = per_season['fold_test_year'].astype(int)
pivot = per_season.pivot_table(
    index=['model_name', 'feature_set'],
    columns='fold_test_year',
    values='brier_score',
)
print("Per-Season Brier Scores:")
print(pivot.round(4).to_string())

In [ ]:
fig = plot_brier_by_season(brier_df, feature_set='full')
plt.tight_layout()
plt.show()

In [ ]:
fig = plot_brier_by_season(brier_df, feature_set='core')
plt.tight_layout()
plt.show()

## Calibration Curves (Reliability Diagrams)
Diagonal line = perfect calibration. Points above diagonal = model is underconfident. Points below = overconfident.

In [ ]:
fig = plot_calibration_curves(results_df, feature_set='full', n_bins=10)
plt.tight_layout()
plt.show()

In [ ]:
fig = plot_calibration_curves(results_df, feature_set='core', n_bins=10)
plt.tight_layout()
plt.show()

## Full vs Core Feature Set Comparison
Does including sp_recent_era_diff (the only column that differs, derived from FanGraphs pitching_stats_range) improve predictions?

In [ ]:
comparison = agg.pivot_table(
    index='model_name',
    columns='feature_set',
    values='brier_score',
).round(4)
comparison['diff (full - core)'] = comparison.get('full', 0) - comparison.get('core', 0)
print("Brier Score: Full vs Core Features")
print("Negative diff = full features are better")
print(comparison.to_string())

## Per-Season Accuracy
Fraction of games where the predicted probability > 0.5 matched the actual outcome.

> **Note:** Accuracy at the 0.5 threshold is a secondary sanity check, not the primary evaluation metric. Brier score is the headline number -- it rewards well-calibrated probabilities across the full distribution, not just binary correct/wrong calls at a fixed threshold.

In [ ]:
results_df['predicted_win'] = (results_df['prob_calibrated'] > 0.5).astype(int)
results_df['correct'] = (results_df['predicted_win'] == results_df['home_win']).astype(int)

acc = results_df.groupby(['model_name', 'feature_set', 'fold_test_year']).agg(
    accuracy=('correct', 'mean'),
    n_games=('correct', 'count'),
).reset_index()
acc_pivot = acc.pivot_table(
    index=['model_name', 'feature_set'],
    columns='fold_test_year',
    values='accuracy',
)
print("Per-Season Accuracy:")
print(acc_pivot.round(3).to_string())

## Summary
Key findings from walk-forward backtest across 5 seasonal folds (2019, 2021-2024).

In [ ]:
best = agg.iloc[0]
print(f"Best model: {best['model_name']} ({best['feature_set']} features)")
print(f"Best aggregate Brier score: {best['brier_score']:.4f}")
print(f"Games evaluated: {best['n_games']}")
print(f"\nAll models compared on {sorted(results_df['fold_test_year'].unique().tolist())} test seasons")
print(f"Calibration applied: isotonic regression (per-fold)")
print(f"Walk-forward: expanding window, no random splits")